# PeakVI (ATAC-only) benchmark — Patient 1 (03H096 / PB2)

**Role in the paper:** Cross-modal benchmark (Extended Data Fig. 7). It asks how much
of the trimodal structure — in particular the Root/CSC cluster — can be
recovered from chromatin accessibility alone.

**What this notebook does**
1. Loads the raw MuData and keeps the barcodes retained by MultiVI
2. Filters the peak matrix (rare peaks and non-chromosomal contigs)
3. Trains **PeakVI** and extracts the latent space
4. Builds neighbours, Leiden clusters and a ForceAtlas2 layout
5. Compares the PeakVI clusters with the trimodal `Cluster_Final` labels and
   highlights the LSC cluster
6. Writes `PeakVI_PB2.h5ad`, consumed by `04_Single_modality_checks/LSC_recovery_Patient1_03H096.ipynb`

**Objects**
- **Reads:** `DATA_DIR / "Teaseq_PB2.h5mu"` and
  `DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned.h5ad"`
- **Creates:** `DATA_DIR / "04_Single_modality_checks/ATAC/PeakVI_PB2.h5ad"`

## Paths and settings

In [ ]:
from pathlib import Path

# Root of the companion data package. Point this at your local copy.
DATA_DIR = Path("PATH_TO_DATA")  # <-- set this to your local data root
OUT_DIR = DATA_DIR / "outputs/PeakVI_PB2"     # figures and tables written by this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import muon
import numpy as np
import pandas as pd
import scanpy as sc
import scvi

## Load the raw MuData

In [ ]:
teapb2 = muon.read(DATA_DIR / "Teaseq_PB2.h5mu")
teapb2.var_names_make_unique()

In [ ]:
# Update indices for teapb2
teapb2.obs.index = [name + '_PB2' for name in teapb2.obs_names]
teapb2.mod['rna'].obs.index = [name + '_PB2' for name in teapb2.mod['rna'].obs_names]
teapb2.mod['atac'].obs.index = [name + '_PB2' for name in teapb2.mod['atac'].obs_names]
teapb2.mod['protein'].obs.index = [name + '_PB2' for name in teapb2.mod['protein'].obs_names]

## Restrict to the QC-passing barcodes

In [ ]:
import muon
import anndata as ad

# Load the cleaned objects
cleaned_teapb2 = ad.read_h5ad(DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned.h5ad")

# Extract the cell indices from the cleaned objects
cleaned_teapb2_cell_ids = cleaned_teapb2.obs.index.tolist()

# Filter the original objects to keep only the cells present in their cleaned counterparts
teapb2 = teapb2[teapb2.obs.index.isin(cleaned_teapb2_cell_ids)]

# Print the dimensions of the filtered objects
print(f"Filtered teapb2 dimensions: {teapb2.shape}")

## Extract the ATAC modality

In [ ]:
adata=teapb2.copy()

In [ ]:
# PeakVI only needs the peak matrix.
atac_adata = adata.mod['atac']
atac_adata.obs.index = adata.obs.index

## Peak filtering

Peaks accessible in fewer than 0.5% of cells are dropped, then the remaining regions are parsed into chromosome / start / end and non-chromosomal contigs are removed.

In [ ]:
print(atac_adata.shape)
# compute the threshold: 0.5% of the cells
min_cells = int(atac_adata.shape[0] * 0.005)
# in-place filtering of regions
sc.pp.filter_genes(atac_adata, min_cells=min_cells)
print(atac_adata.shape)

In [ ]:
# Parse peak coordinates from the "chr:start-end" style feature id.
split_interval = atac_adata.var["gene_ids"].str.split(":", expand=True)
atac_adata.var["chr"] = split_interval[0]
split_start_end = split_interval[1].str.split("-", expand=True)
atac_adata.var["start"] = split_start_end[0].astype(int)
atac_adata.var["end"] = split_start_end[1].astype(int)


In [ ]:
# Filter out non-chromosomal regions
mask = atac_adata.var["chr"].str.startswith("chr")
atac_adata = atac_adata[:, mask].copy()

## Train PeakVI

In [ ]:
adata=atac_adata.copy()

In [ ]:
scvi.model.PEAKVI.setup_anndata(adata)

In [ ]:
# Default PeakVI architecture and training schedule.
model = scvi.model.PEAKVI(adata)
model.train()

## Latent space, clustering and ForceAtlas2 layout

In [ ]:
PEAKVI_LATENT_KEY = "X_peakvi"
adata.obsm[PEAKVI_LATENT_KEY] = model.get_latent_representation()

# 2. NEIGHBORS + GRAPH + CLUSTERING
PEAKVI_CLUSTERS_KEY = "clusters_peakvi"

sc.pp.neighbors(adata, use_rep=PEAKVI_LATENT_KEY)
sc.tl.draw_graph(adata, layout="fa")
sc.tl.leiden(adata, key_added=PEAKVI_CLUSTERS_KEY, resolution=0.8)

# 3. EMBEDDING PLOT
muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color=[PEAKVI_CLUSTERS_KEY],
    frameon=False,
    ncols=1,
)

## Compare the PeakVI clusters with the trimodal labels

In [ ]:
# Trimodal reference labels, transferred for visual comparison only.
data = ad.read_h5ad(DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad")
adata.obs['Cluster_Final'] = data.obs['Cluster_Final']


muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color=["clusters_peakvi", "Cluster_Final"],
    frameon=False,
    ncols=2,
)

## Highlight the LSC cluster

In [ ]:
cluster_to_highlight = "6"

import numpy as np
import pandas as pd

adata.obs["highlight"] = np.where(
    adata.obs["Cluster_Final"] == cluster_to_highlight,
    cluster_to_highlight,
    "Other"
)

# Draw the highlighted cluster on top of the grey background.
adata.obs["highlight"] = pd.Categorical(
    adata.obs["highlight"],
    categories=["Other", cluster_to_highlight],
    ordered=True
)

adata.uns["highlight_colors"] = [
    "#D3D3D3",  # Other → grey
    "#FF0000"   # Highlight → red
]

muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color="highlight",
    size=50,
    frameon=False,
)

## Save the ATAC embedding

In [ ]:
target_dir = DATA_DIR / "04_Single_modality_checks/ATAC"
target_dir.mkdir(parents=True, exist_ok=True)

output_path = target_dir / "PeakVI_PB2.h5ad"
adata.write(output_path)

print("Saved:", output_path, "|", output_path.is_file())